# Verify processed dataset
Run after post-processing, from the directory containing `processed_data/`, `aztc_full_localdoc/`, and `near_duplicated_data_post/`. Requires the existing FastText installation and `/tmp/lid.176.bin`.

Checks all saved rows without changing any data files. A failed assertion identifies a failed check. Near-duplicate verification checks the saved MinHash results from the same processing run; it does not repeat pair discovery or guarantee exhaustive detection. Documents with fewer than 10 normalized words remain outside that check, as in post-processing.

In [1]:
import os
from pathlib import Path
import fasttext
from pyspark.sql import SparkSession, functions as F

os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 10g pyspark-shell"
spark = SparkSession.builder.appName("verify-dataset").master("local[*]").getOrCreate()
DATA = Path("processed_data")
BASE = "near_duplicated_data_post"
model_path = Path("/tmp/lid.176.bin")

## Check shard names, readability, and schema

In [2]:
files = sorted(DATA.glob("train-*-of-*.parquet"))
assert files, "No processed train shards found"
assert [p.name for p in files] == [
    f"train-{i:05d}-of-{len(files):05d}.parquet" for i in range(len(files))
], "Shard numbering is incomplete or inconsistent"
dataset = spark.read.option("mergeSchema", "true").parquet(*map(str, files)).persist()
assert dataset.dtypes == [("text", "string"), ("source", "string")], "Unexpected output schema"
print(f"Verified shard names and schema: {len(files)} files")

Verified shard names and schema: 1 files


## Check row count and content rules
Violation counts must all be zero. The disallowed-character check includes Cyrillic.

In [3]:
conditions = {
    "Rows with non-ANL or null source": ~F.col("source").eqNullSafe("ANL"),
    "Rows with null or empty text": F.col("text").isNull() | (F.col("text") == ""),
    "Rows with whitespace-only text": F.col("text").rlike(r"(?U)^\s+$"),
    "Rows with disallowed characters": F.col("text").rlike(r"[^\p{IsLatin}\p{N}\p{P} \t\r\n\f]"),
    "Rows shorter than 500 characters": F.length("text") < 500,
}
counts = dataset.agg(
    F.count("*").alias("Saved rows"),
    *(F.count(F.when(condition, 1)).alias(label) for label, condition in conditions.items())
).first().asDict()
for label, count in counts.items():
    print(f"{label}: {count:,}")
assert counts["Saved rows"] > 0, "Processed dataset is empty"
assert not any(counts[label] for label in conditions), "Content-rule violations found"

Saved rows: 208,293
Rows with non-ANL or null source: 0
Rows with null or empty text: 0
Rows with whitespace-only text: 0
Rows with disallowed characters: 0
Rows shorter than 500 characters: 0


## Check exact duplicates and original-text preservation

In [4]:
duplicates = counts["Saved rows"] - dataset.select("text").distinct().count()
print(f"Redundant exact-duplicate rows: {duplicates:,}")
assert duplicates == 0, "Exact duplicates remain"
originals = spark.read.parquet("aztc_full_localdoc/train-*-of-*.parquet").filter(F.col("source") == "ANL")
unmatched = dataset.join(originals.select("text", "source"), ["text", "source"], "left_anti").count()
print(f"Saved rows absent from original ANL data: {unmatched:,}")
assert unmatched == 0, "Some saved texts or sources differ from the original data"

Redundant exact-duplicate rows: 0
Saved rows absent from original ANL data: 0


## Recheck Azerbaijani classification
Loads the existing local model once per partition and classifies every saved document.

In [5]:
def count_non_azerbaijani(rows):
    model = fasttext.load_model(str(model_path))
    yield sum(model.predict(" ".join(row.text.split()))[0][0] != "__label__az" for row in rows)

non_azerbaijani = dataset.select("text").rdd.mapPartitions(count_non_azerbaijani).sum()
print(f"Rows classified other than Azerbaijani: {non_azerbaijani:,}")
assert non_azerbaijani == 0, "Non-Azerbaijani classifications remain"

Rows classified other than Azerbaijani: 0


## Verify removal against saved MinHash results
Matching pairs and distinct documents marked for removal are different counts. Compare the expected eligible survivors with the eligible rows actually saved; both differences must be zero. This also checks for missing eligible rows.

In [6]:
pairs = spark.read.parquet(f"{BASE}/pairs").persist()
removed = pairs.select(F.col("id_b").alias("id")).distinct().persist()
expected_ids = spark.read.parquet(f"{BASE}/features").select("id").join(removed, "id", "left_anti").persist()
actual_ids = dataset.filter(F.size(F.split(
    F.trim(F.regexp_replace(F.lower("text"), r"[^\p{L}\p{N}]+", " ")), r"\s+"
)) >= 10).select(F.sha2("text", 256).alias("id")).persist()

print(f"Saved near-duplicate matches: {pairs.count():,} pairs")
print(f"Distinct documents marked for removal: {removed.count():,}")
print(f"Expected eligible rows after removal: {expected_ids.count():,}")
print(f"Eligible rows in saved dataset: {actual_ids.count():,}")
missing = expected_ids.exceptAll(actual_ids).count()
unexpected = actual_ids.exceptAll(expected_ids).count()
print(f"Expected eligible rows missing from output: {missing:,}")
print(f"Unexpected eligible rows in output: {unexpected:,}")
assert missing == 0 and unexpected == 0, "Saved output does not match the MinHash removal results"

for cached in (dataset, pairs, removed, expected_ids, actual_ids):
    cached.unpersist()
print("PASS: all dataset verification checks passed")

Saved near-duplicate matches: 3,454 pairs
Distinct documents marked for removal: 2,982
Expected eligible rows after removal: 208,293
Eligible rows in saved dataset: 208,293
Expected eligible rows missing from output: 0
Unexpected eligible rows in output: 0
PASS: all dataset verification checks passed
